# 77 Unity Catalog · Lineage, Auditoría

Vas a generar lineage deliberadamente, consultar sus system tables cuando estén disponibles, ubicar los audit logs y documentar PII como metadato gobernado. 

In [0]:
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA}")

In [0]:
# Declara los tres objetos que formarán el grafo de lineage y verifica la tabla de origen.
TABLA_PADRON = f"{CATALOG}.{SCHEMA}.padron"
TABLA_AGREGADO = f"{CATALOG}.{SCHEMA}.padron_agregado_provincia"
VISTA_TOP = f"{CATALOG}.{SCHEMA}.v_padron_top_provincias"
assert spark.catalog.tableExists(TABLA_PADRON), (
    f"No existe {TABLA_PADRON}. Ejecutá primero el notebook 75_UC_Jerarquia_y_Permisos.")

# Materializa un agregado por provincia; esta lectura registra el primer salto de lineage.
spark.sql(f"""
CREATE OR REPLACE TABLE {TABLA_AGREGADO}
AS SELECT provincia, COUNT(*) AS personas_inscritas
FROM {TABLA_PADRON}
GROUP BY provincia
""")
# Crea una vista de consumo sobre el agregado para completar el segundo salto del grafo.
spark.sql(f"""
CREATE OR REPLACE VIEW {VISTA_TOP}
AS SELECT provincia, personas_inscritas
FROM {TABLA_AGREGADO}
ORDER BY personas_inscritas DESC
""")
display(spark.table(VISTA_TOP))

## Inspección visual del lineage

1. Abrí **Catalog Explorer** y buscá `padron_agregado_provincia`.
2. Entrá a la pestaña **Lineage** y elegí **See Lineage Graph**.
3. Expandí los nodos hasta observar `padron → padron_agregado_provincia → v_padron_top_provincias`.
4. Hacé clic en `provincia` o `personas_inscritas` para revisar el *column-level lineage*.

El grafo puede mostrar tablas, views, notebooks, jobs, pipelines, queries, dashboards y versiones de modelos. El lineage permite seguir impacto y procedencia sin leer manualmente todo el código.

## Lineage programático con fallback

Las system tables pueden no estar habilitadas en Free Edition. La consulta intenta leer el historial y, si no tiene acceso, ofrece un fallback funcional mediante `information_schema.views`.

In [0]:
# Intenta consultar el lineage centralizado cuando las system tables están habilitadas.
try:
    df_lineage = spark.sql(f"""
        SELECT source_table_full_name, target_table_full_name, entity_type, event_time
        FROM system.access.table_lineage
        WHERE target_table_schema = '{SCHEMA}'
        ORDER BY event_time DESC
        LIMIT 20
    """)
    display(df_lineage)
except Exception as e:
    # Si no hay permisos o habilitación, usa las definiciones de views como evidencia alternativa.
    print("System tables no disponibles en este workspace:", e)
    print("Fallback: usá Catalog Explorer > Lineage, o information_schema.")
    display(spark.sql(f"""
        SELECT table_name, view_definition
        FROM {CATALOG}.information_schema.views
        WHERE table_schema = '{SCHEMA}'
        ORDER BY table_name
    """))

## Dónde viven los audit logs

Los audit logs se almacenan en `system.access.audit`, una tabla del catálogo `system` gobernada por Unity Catalog, con retención gratuita de **365 días**. Los eventos de nivel workspace son regionales y los eventos de nivel cuenta son globales. Sus tablas hermanas incluyen `system.access.table_lineage` y `system.access.column_lineage`.

Habilitar system tables requiere un **account admin**; por eso pueden no estar accesibles en Free Edition. Si la consulta siguiente cae al fallback, revisá la actividad desde la UI y conservá la consulta como referencia para un workspace habilitado.

In [0]:
# Consulta eventos recientes de Unity Catalog relacionados con grants y operaciones de tablas.
try:
    df_auditoria = spark.sql("""
        SELECT event_time, user_identity.email AS usuario, action_name, request_params
        FROM system.access.audit
        WHERE service_name = 'unityCatalog'
          AND (lower(action_name) LIKE '%grant%' OR lower(action_name) LIKE '%table%')
        ORDER BY event_time DESC
        LIMIT 20
    """)
    display(df_auditoria)
except Exception as e:
    # El fallback muestra privilegios declarados cuando el audit log no está disponible.
    print("Audit system table no disponible en este workspace:", e)
    print("Fallback: revisá Catalog Explorer > Lineage y los grants en information_schema.")
    display(spark.sql(f"""
        SELECT grantor, grantee, table_name, privilege_type
        FROM {CATALOG}.information_schema.table_privileges
        WHERE table_schema = '{SCHEMA}'
        ORDER BY table_name, grantee
    """))

## Metadatos como gobernanza

Los comentarios hacen visible el propósito y la sensibilidad de los datos en Catalog Explorer. Son una práctica manual; los *governed tags* de ABAC permiten convertir esa clasificación en políticas centralizadas y ejecutables.

In [0]:
# Documenta el propósito y la sensibilidad de la tabla en los metadatos de Unity Catalog.
spark.sql(f"""
COMMENT ON TABLE {TABLA_PADRON} IS
'Padrón electoral público del TSE; contiene PII y requiere uso proporcional, trazable y autorizado'
""")
# Mantiene la clasificación por columna en un diccionario para aplicarla de forma uniforme.
comentarios_pii = {
    "cedula": "PII directa: identificador único de la persona",
    "nombre": "PII: nombre de pila",
    "apellido1": "PII: primer apellido",
    "apellido2": "PII: segundo apellido",
    "nombre_completo": "PII directa: nombre completo de la persona",
    "provincia": "Ubicación electoral agregable; puede aumentar riesgo de reidentificación",
    "canton": "Ubicación electoral; dato cuasi-identificador",
    "distrito": "Ubicación electoral granular; dato cuasi-identificador"
}
# Actualiza cada comentario sin modificar los valores almacenados en la tabla.
for columna, comentario in comentarios_pii.items():
    spark.sql(f"ALTER TABLE {TABLA_PADRON} ALTER COLUMN {columna} COMMENT '{comentario}'")

# Comprueba desde information_schema que la documentación quedó registrada.
display(spark.sql(f"""
SELECT column_name, data_type, comment
FROM {CATALOG}.information_schema.columns
WHERE table_schema = '{SCHEMA}' AND table_name = 'padron'
ORDER BY ordinal_position
"""))